# 3.12 Using the model

এই লেসনে আমরা আমাদের প্রজেক্টের একদম শেষ ধাপে আছি। এখানে আমরা সম্পূর্ণ ট্রেইনিং ডেটার (train + validation) ওপর মডেলটিকে নতুন করে ট্রেইন করব, টেস্ট সেটের (test set) ওপর এর পারফরম্যান্স চেক করব এবং একজন কাস্টমারের Churn probability বের করে দেখাব।

### কিছু গুরুত্বপূর্ণ Technical Terms (শব্দার্থ):
* **Full Training Data:** Training Data এবং Validation Data একত্রে।
* **Test Set:** একদম নতুন কিছু ডেটা, যা মডেল আগে কখনো দেখেনি। এটি দিয়ে মডেলের আসল দক্ষতা মাপা হয়।
* **Overfitting:** যখন মডেল ট্রেইনিং ডেটা মুখস্থ করে ফেলে, কিন্তু নতুন ডেটায় (টেস্ট সেটে) ভালো ফলাফল দিতে পারে না।

## Final model (চূড়ান্ত মডেল ট্রেইন করা)

এতক্ষণ মডেলের parameter tune করার জন্য আমরা validation set ব্যবহার করছিলাম। এখন যেহেতু tuning শেষ, তাই আমরা training এবং validation ডেটাকে একসাথে করে (`df_full_train`) মডেল ট্রেইন করব। ডেটা যত বেশি হবে, মডেল তত ভালো শিখবে।

In [ ]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

# সম্পূর্ণ ট্রেইনিং ডেটাকে ডিকশনারিতে কনভার্ট করা
dicts_full_train = df_full_train[categorical + numerical].to_dict(orient='records')

# নতুন করে DictVectorizer ট্রেইন এবং ডেটা ট্রান্সফর্ম করা
dv = DictVectorizer(sparse=False)
X_full_train = dv.fit_transform(dicts_full_train)

# টার্গেট ভ্যারিয়েবল (Churn)
y_full_train = df_full_train.churn.values

# মডেল ট্রেইন করা
model = LogisticRegression(solver='lbfgs')
model.fit(X_full_train, y_full_train)

**মনে রাখবে:** যখনই মডেল নতুন ডেটায় ট্রেইন করবে, তখন `DictVectorizer`-কেও নতুন ডেটার ওপর `fit` করতে হবে। Encoding এবং Model সবসময় একসাথে কাজ করে।

## Checking on the test set (টেস্ট সেটে পারফরম্যান্স চেক করা)

এবার আমরা মডেলকে test set দিয়ে পরীক্ষা করব।

In [ ]:
# টেস্ট সেটকে ডিকশনারিতে কনভার্ট করা
dicts_test = df_test[categorical + numerical].to_dict(orient='records')

# আগের DictVectorizer দিয়েই transform করা (এখানে fit_transform হবে না)
X_test = dv.transform(dicts_test)

# Probability প্রেডিক্ট করা
y_pred = model.predict_proba(X_test)[:, 1]

# 50% এর বেশি সম্ভাবনা থাকলে Churn (1), অন্যথায় No Churn (0)
churn_decision = (y_pred >= 0.5)

# Accuracy চেক করা
print("Test Accuracy:", (churn_decision == y_test).mean())

এখানে আমরা Accuracy পাবো প্রায় 81.5%। আগের লেসনে validation সেটে Accuracy ছিল 80.3%।
যেহেতু দুটি Accuracy কাছাকাছি, এর মানে হলো মডেলটি নতুন ডেটায় (unseen data) ভালো কাজ করছে এবং কোনো **Overfitting** হয়নি।

## Scoring one customer (একজন কাস্টমারের ওপর মডেল প্রয়োগ)

বাস্তবে (Production-এ) আমরা একসাথে পুরো টেস্ট সেট প্রেডিক্ট করি না, বরং একজন একজন কাস্টমারের ডেটা আসে এবং মডেল তার ওপর প্রেডিকশন দেয়।

চলো, টেস্ট সেটের একদম শেষ কাস্টমারের ওপর মডেল প্রয়োগ করে দেখি।

In [ ]:
# টেস্ট সেটের শেষ কাস্টমারের ডেটা নেওয়া
customer = dicts_test[-1]
customer

DictVectorizer সবসময় ডেটার লিস্ট এক্সপেক্ট করে, তাই আমরা কাস্টমারের ডিকশনারিকে লিস্টের `[ ]` ভেতর দিয়ে পাস করব।

In [ ]:
# কাস্টমারের ডেটা এনকোড করা
X_small = dv.transform([customer])

# Probability বের করা
prob = model.predict_proba(X_small)[0, 1]
print("Churn Probability:", prob)

মডেল বলছে এই কাস্টমারের সার্ভিস ছেড়ে যাওয়ার (Churn) সম্ভাবনা প্রায় 60% (0.596)। যেহেতু এটি আমাদের 0.5 threshold-এর ওপরে, তাই মডেলের সিদ্ধান্ত হলো- **"এই কাস্টমার Churn করবে"**।

চলো আসল ডেটার সাথে মিলিয়ে দেখি মডেলের কথা সত্য কিনা:

In [ ]:
# আসল ডেটায় কাস্টমারের স্ট্যাটাস কী ছিল?
y_test[-1]

আউটপুট `1`, তার মানে এই কাস্টমার সত্যিই সার্ভিস ছেড়ে চলে গিয়েছিল! আমাদের মডেল সঠিক প্রেডিকশন দিয়েছে।

বাস্তব প্রজেক্টে এভাবেই কাজ করা হয়। যখন কোনো কাস্টমারের Churn Probability অনেক বেশি দেখা যায়, তখন কোম্পানি তাকে ধরে রাখার জন্য প্রমোশনাল ইমেইল বা ডিসকাউন্ট অফার পাঠায়।